# Blindfold SDK — PII Protection for AI Applications

[![PyPI](https://img.shields.io/pypi/v/blindfold-sdk)](https://pypi.org/project/blindfold-sdk/)
[![License](https://img.shields.io/badge/license-MIT-blue.svg)](https://opensource.org/licenses/MIT)

**Detect, tokenize, redact, mask, hash, and synthesize PII — all running locally, no API key needed.**

Blindfold is a developer-first SDK for protecting personally identifiable information (PII) in AI pipelines. This notebook demonstrates every feature using **local mode** — everything runs on your machine with zero external calls.

- [Documentation](https://docs.blindfold.dev)
- [GitHub](https://github.com/blindfold-dev)
- [Website](https://blindfold.dev)

## 1. Install & Setup

In [ ]:
%pip install -q blindfold-sdk

In [ ]:
from blindfold import Blindfold

# No API key = local mode (regex-based detection, runs entirely on your machine)
blindfold = Blindfold()

print("Blindfold initialized in local mode")

## 2. Detect PII

Find PII in text without modifying it. Returns entity types, positions, and confidence scores.

In [ ]:
text = """Contact John at john.doe@acme.com or call +1 (555) 867-5309.
His SSN is 123-45-6789 and credit card is 4111-1111-1111-1111.
Send payments to IBAN DE89370400440532013000."""

result = blindfold.detect(text)

print(f"Found {result.entities_count} PII entities:\n")
print(f"{'Type':<25} {'Value':<35} {'Position':<12} {'Score'}")
print("-" * 80)
for entity in result.detected_entities:
    print(f"{entity.type:<25} {entity.text:<35} {entity.start}-{entity.end:<8} {entity.score:.2f}")

## 3. Tokenize & Detokenize (Reversible)

Replace PII with safe tokens, send to any LLM, then restore the original values. **PII never leaves your system.**

In [ ]:
text = "Please update the account for Sarah Connor (sarah@skynet.com, +1-555-123-4567) with SSN 321-54-9876."

tokenized = blindfold.tokenize(text)

print("=== Tokenized Text (safe to send to LLM) ===")
print(tokenized.text)
print("\n=== Token Mapping (stays on your server) ===")
for token, original in tokenized.mapping.items():
    print(f"  {token} -> {original}")

In [ ]:
# Simulate an LLM response that uses the same tokens
llm_response = f"I've updated the account for the customer. " \
    f"A confirmation email has been sent to {list(tokenized.mapping.keys())[0]}."

print("=== LLM Response (contains tokens) ===")
print(llm_response)

# Detokenize — restore original values (runs locally, no API call)
restored = blindfold.detokenize(llm_response, tokenized.mapping)

print("\n=== Restored Response (real values back) ===")
print(restored.text)

## 4. Redact (Irreversible)

Permanently remove PII — useful for logs, analytics, or any data that shouldn't contain personal information.

In [ ]:
text = "Patient Jane Smith (jane.smith@hospital.org, SSN 234-56-7890) was diagnosed on 2024-03-15."

# Redact all detected PII
result = blindfold.redact(text)
print("=== Full Redaction ===")
print(result.text)
print(f"Entities redacted: {result.entities_count}")

# Selective redaction — only remove specific entity types
result_selective = blindfold.redact(text, entities=["Email Address", "Social Security Number"])
print("\n=== Selective Redaction (email + SSN only) ===")
print(result_selective.text)

## 5. Mask, Hash, and Synthesize

Three more ways to protect PII, each suited for different use cases.

In [ ]:
text = "Contact support: alice@example.com, phone +1-555-987-6543, card 4532-1234-5678-9012."

# Mask — show last 4 characters, hide the rest
masked = blindfold.mask(text, chars_to_show=4, from_end=True)

# Hash — deterministic replacement for analytics
hashed = blindfold.hash(text)

# Synthesize — replace with realistic fake data
synthesized = blindfold.synthesize(text)

print("=== Comparison of All Methods ===\n")
methods = [
    ("Original", text),
    ("Masked", masked.text),
    ("Hashed", hashed.text),
    ("Synthesized", synthesized.text),
]

for name, result_text in methods:
    print(f"{name}:")
    print(f"  {result_text}\n")

In [ ]:
# Hash produces the same output for the same input — useful for tracking
text_a = "Email me at alice@example.com"
text_b = "Forward this to alice@example.com"

hash_a = blindfold.hash(text_a)
hash_b = blindfold.hash(text_b)

print("Hash is deterministic — same email produces same hash:")
print(f"  Text A: {hash_a.text}")
print(f"  Text B: {hash_b.text}")

## 6. Policies

Built-in policies control which entity types to detect. Compare how different compliance policies handle the same text.

In [ ]:
text = """Patient record: John Smith, DOB 1990-05-15, SSN 123-45-6789.
Contact: john@hospital.org, +1-555-234-5678.
Card on file: 4111-1111-1111-1111, IBAN DE89370400440532013000.
Diagnosis: hypertension. Prescribed lisinopril 10mg."""

policies = ["basic", "gdpr_eu", "hipaa_us", "pci_dss", "strict"]

print(f"{'Policy':<12} {'Entities Found':<18} {'Types Detected'}")
print("-" * 75)

for policy in policies:
    result = blindfold.detect(text, policy=policy)
    types = sorted(set(e.type for e in result.detected_entities))
    print(f"{policy:<12} {result.entities_count:<18} {', '.join(types)}")

## 7. Multi-Locale Support

Detect PII formats from different countries. Configure locales at initialization.

In [ ]:
# Initialize with multiple locales
bf_multi = Blindfold(locales=["us", "eu"])

texts = {
    "US format": "SSN: 123-45-6789, phone: +1 (555) 867-5309",
    "EU format": "IBAN: DE89370400440532013000, phone: +49 30 901820",
    "Mixed": "US card 4111-1111-1111-1111, EU IBAN FR7630006000011234567890189",
}

for label, text in texts.items():
    result = bf_multi.detect(text)
    types = [e.type for e in result.detected_entities]
    print(f"{label}: {result.entities_count} entities -> {types}")

## 8. Batch Processing

Process multiple texts efficiently. In local mode, loop over texts; with an API key, use `tokenize_batch()` for server-side batching.

In [ ]:
import time

support_tickets = [
    "Hi, my name is Alice and my email is alice@corp.com. Order #12345 is wrong.",
    "I'm Bob, call me at +1-555-111-2222. My account shows incorrect charges.",
    "Card ending 4111-1111-1111-1111 was charged twice. Contact me at charlie@mail.com.",
    "Please update my address. SSN for verification: 111-22-3333. Thanks, Diana.",
    "My IBAN DE89370400440532013000 wasn't credited. Reach me at +49 170 1234567.",
]

start = time.perf_counter()
results = [blindfold.tokenize(ticket) for ticket in support_tickets]
elapsed = time.perf_counter() - start

print(f"Processed {len(results)} tickets in {elapsed:.3f}s\n")

for i, tokenized in enumerate(results):
    print(f"Ticket {i + 1}: {tokenized.text[:80]}...")
    print(f"  Tokens: {list(tokenized.mapping.keys())}\n")

## 9. LLM Integration Pattern (Simulated)

The full flow: **user message with PII -> tokenize -> send to LLM -> detokenize -> safe response**.

No API keys needed — we mock the LLM to show the pattern.

In [ ]:
def mock_llm(prompt: str) -> str:
    """Simulates an LLM response. In production, replace with OpenAI/Anthropic/etc."""
    return (
        "Based on the information provided, I've located your account. "
        "I can see that your recent order was shipped to the address on file. "
        "A confirmation has been sent to your email. "
        "Is there anything else I can help you with?"
    )


# User sends a message containing PII
user_message = """Hi, I'm Sarah Connor. My email is sarah@skynet.com and my phone
is +1-555-999-8888. My SSN is 321-54-9876. I need to check my order status."""

print("STEP 1: User message (contains PII)")
print(f"  {user_message}\n")

# Step 1: Tokenize — PII is replaced with safe tokens
tokenized = blindfold.tokenize(user_message)
print("STEP 2: Tokenized (safe to send to LLM)")
print(f"  {tokenized.text}\n")

# Step 2: Send tokenized text to LLM
llm_response = mock_llm(tokenized.text)
print("STEP 3: LLM response (no real PII exposed)")
print(f"  {llm_response}\n")

# Step 3: Detokenize — restore original values in the response
final = blindfold.detokenize(llm_response, tokenized.mapping)
print("STEP 4: Final response (PII restored for the user)")
print(f"  {final.text}\n")

print("The LLM never saw the real PII. The mapping stayed on your server.")

## 10. Performance Benchmark

Blindfold's local mode uses an optimized regex scanner. Let's measure throughput.

In [ ]:
import time

# Generate sample texts
sample_texts = [
    f"User {i}: email user{i}@company.com, phone +1-555-{i:03d}-{(i*7)%10000:04d}, SSN {i%900+100}-{i%90+10}-{i%9000+1000}"
    for i in range(1000)
]

# Benchmark detect
start = time.perf_counter()
for text in sample_texts:
    blindfold.detect(text)
detect_time = time.perf_counter() - start

# Benchmark tokenize
start = time.perf_counter()
for text in sample_texts:
    blindfold.tokenize(text)
tokenize_time = time.perf_counter() - start

# Benchmark redact
start = time.perf_counter()
for text in sample_texts:
    blindfold.redact(text)
redact_time = time.perf_counter() - start

print("=== Blindfold Performance (1,000 texts) ===\n")
print(f"{'Method':<12} {'Total (s)':<12} {'Texts/sec':<12}")
print("-" * 36)
print(f"{'Detect':<12} {detect_time:<12.3f} {1000/detect_time:<12,.0f}")
print(f"{'Tokenize':<12} {tokenize_time:<12.3f} {1000/tokenize_time:<12,.0f}")
print(f"{'Redact':<12} {redact_time:<12.3f} {1000/redact_time:<12,.0f}")

In [ ]:
# Optional: Compare with Presidio (if installed)
try:
    from presidio_analyzer import AnalyzerEngine

    analyzer = AnalyzerEngine()

    start = time.perf_counter()
    for text in sample_texts[:100]:
        analyzer.analyze(text=text, language="en")
    presidio_time = time.perf_counter() - start

    # Blindfold on same subset
    start = time.perf_counter()
    for text in sample_texts[:100]:
        blindfold.detect(text)
    bf_time = time.perf_counter() - start

    speedup = presidio_time / bf_time
    print(f"\n=== Blindfold vs Presidio (100 texts) ===")
    print(f"Presidio:  {presidio_time:.3f}s ({100/presidio_time:,.0f} texts/sec)")
    print(f"Blindfold: {bf_time:.3f}s ({100/bf_time:,.0f} texts/sec)")
    print(f"Speedup:   {speedup:.0f}x faster")
except ImportError:
    print("Presidio not installed — skipping comparison.")
    print("Install with: pip install presidio-analyzer presidio-anonymizer")
    print("\nBased on our benchmarks, Blindfold is ~56x faster than Presidio")
    print("with higher detection accuracy (F1: 71.8% vs 49.2%).")

## Next Steps

**Local mode** (what you just used) runs entirely on your machine with regex-based detection — great for speed and privacy.

**Cloud mode** adds NLP-powered detection for higher accuracy on names, addresses, and contextual PII:

```python
blindfold = Blindfold(api_key="your-api-key")
```

Get a free API key at [app.blindfold.dev](https://app.blindfold.dev)

### Resources

- [Documentation](https://docs.blindfold.dev)
- [Cookbook Examples](https://github.com/blindfold-dev/blindfold-cookbook)
- [Python SDK on PyPI](https://pypi.org/project/blindfold-sdk/)
- [Node.js SDK on npm](https://www.npmjs.com/package/@blindfold/sdk)